In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# A：复用四张已生成图的回正诊断（不重新生成）

已填入diagnostic-20260909-152107-462244四图目录。0新生成；仅加载原方法的VAE和内容评分资产，8次公共reader、32次固定17-key内容评分（完整成功时40次VAE编码）。需要Colab GPU及现有Secrets HF_TOKEN、CEG_WM_ROOT_KEY。

每图：clean原样/identity实际采样/估计回正；+10攻击原样/真实H回正/估计回正/真实角±1°回正，共8观察。plain作为负样本、content_only作为无锚点对照，四图全部保留。各回正均从同一原图或攻击图独立单次warp；±1°只是粗略容差探针，不是精确角度阈值。oracle不参与几何选择或正式检测，完整路径只比较pre与估计post。

真实攻击中心255与reader映射中心255.5分别记录；此诊断不能分离历史step18注入与余下step19的影响，只检查最终RGB/VAE再编码及RGB回正。报告原始分支分数和回正损伤，不能以max掩盖失败。输出写入Drive独立reuse-时间目录；无需修改参数。


In [ ]:
import json, os, pathlib, subprocess, sys, datetime
from importlib.metadata import version, PackageNotFoundError

REPO='https://github.com/RICHAAARC/CEG-WM.git'
BRANCH='dev/latent-sync-v1'
EXPECTED_EXACT='b0cade92f0a1d749f6f70292f135e4b4c5115d13'
checkout=pathlib.Path('/content/latent-sync-v1-github')
drive_root=pathlib.Path('/content/drive/MyDrive/CEG-WM/development/latent-sync-v1')
INPUT=drive_root/'diagnostic-20260909-152107-462244'
output_root=drive_root/('reuse-'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
if not checkout.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'fetch','origin',BRANCH],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
subprocess.run([sys.executable,'-m','pip','install','diffusers<0.40','transformers','accelerate','scipy','sentencepiece'],check=True)
environment={}
for package in ('torch','diffusers','numpy','scipy'):
    try: environment[package]=version(package)
    except PackageNotFoundError: environment[package]=None
print({'branch':BRANCH,'code':EXPECTED_EXACT,'versions':environment})
child_env=dict(os.environ)
child_env['PYTHONPATH']=str(checkout/'src')+os.pathsep+str(checkout)
from google.colab import userdata
child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''
child_env['CEG_WM_ROOT_KEY']=userdata.get('CEG_WM_ROOT_KEY') or ''
command=[sys.executable,'-m','experiments.run_latent_sync_reuse_diagnostic',
         '--execute','--input-dir',str(INPUT),'--output',str(output_root)]
completed=subprocess.run(command,cwd=checkout,env=child_env,check=False)
print('输出目录:', output_root)
if completed.returncode != 0:
    raise RuntimeError(f'诊断退出码 {completed.returncode}；已写入结果保留在 {output_root}')
report_path=output_root/'report.json'
if report_path.exists():
    report=json.loads(report_path.read_text())
    print({'report':str(report_path),'summary':report.get('results',report.get('path_comparisons'))})
